In [ ]:
import pandas as pd
import numpy as np
from tomlkit import date

## 案例：学生成绩分析

创建包含 10 名学生数学成绩的 Series，完成统计分析。涉及 `mean()`、`max()`、`min()` 和布尔索引筛选。


In [ ]:
arr = np.random.randint(50,101,10)
s = pd.Series(arr,index=['学生'+str(i) for i in range(1,11)])
print(s,'\n')
print(s.max())
print(s.min())
print(s.mean())
print(s[s.values>s.mean()].count())

## 案例：城市温度分析

对温度 Series 做筛选（高于 30 天）、排序、差分计算。`diff()` 是时间序列相邻差的好工具。


In [ ]:
ind = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
tempe = np.random.randint(20,41,7)
s = pd.Series(tempe,index=ind)
print(s,'\n')
print('温度高于30的天数:',s[s.values>30].count())
print('平均气温:',s.mean())
sort_s = s.sort_values(ascending=False).index.tolist() #tolist()去掉index dtype
print('排序:',*(sort_s)) #*去掉中括号

changes = list()
d = list()

for i in range(len(ind)-1):
    changes.append(abs(tempe[i+1]-tempe[i]))

for j in range(len(ind)-1):
    d.append(ind[j+1]+','+ind[j])

s0 = pd.Series(changes,index=d)
print(s0.idxmax())
#print(s0[s0.values==s0.max()].keys())

tempe0 = s.diff().abs()
print(tempe0)

## 案例：股票收益率计算

`pct_change()` 计算逐日收益率比手算 `(今日-昨日)/昨日` 更简洁。结合 `idxmax()` / `idxmin()` 直接定位极值日期。


In [ ]:
dates = pd.date_range('2026-05-18', periods=10, freq='B') # B代表工作日 自动跳过周末
prices = [100, 102, 101, 105, 104, 108, 110, 109, 112, 115]
s = pd.Series(prices,index=dates)

pp = [np.nan]
for i in range(1,len(dates)):
    pp.append((prices[i]/prices[i-1])-1)

s0 = pd.Series(pp,index=dates,name='收益率') #pp = s.pct_change() 计算百分率变化
print(s0)

print('最高:',s0.idxmax())
print('最低:',s0.idxmin())

print('波动率:',s0.std())

## 案例：月度销售环比分析

`pct_change()` + 重采样 `/` 分组聚合。`rolling()` 滑动窗口可以检测连续增长趋势。

```python
# 月环比增长率：当月相对于上月的增长比例
s.pct_change()        # (本月-上月) / 上月
s.diff()              # 本月 - 上月（绝对差值）
```


In [ ]:
dates = pd.date_range('2025-01-01', periods=12, freq='MS') # MS代表每月第一天 ME代表每月最后一天
sales = pd.Series([
    120, 135, 180,
    210, 190, 160,
    110, 105, 140,
    165, 195, 220
], index=dates, name='销量')

print(sales.resample('QS').mean()) #resample重新采样

print('销量最多为%d月'%sales.idxmax().month)

print(sales.pct_change())

temp = sales.pct_change()
mask = temp.rolling(window=3, min_periods=3).min() > 0 #默认min_period=windows
#min_period: 在窗口框定的存在元素（去掉nan）个数达到min_period的时候窗口才返回这个切片
#rolling()滚动窗口是向前看而非向后看
print(*(temp[mask].keys().month.tolist()))

## 补充：DateOffset 时间频率常量

在 `pd.date_range(freq=...)` 中使用：

| 代码 | 含义 | 示例 |
|---|---|---|
| `D` | 自然日 | 每天，含周末 |
| `B` | 工作日 | 跳过周末 |
| `W` | 每周（周日为末） | 每周日采样 |
| `M` | 月末 | 每月最后一个自然日 |
| `MS` | 月初 | 每月第一天 |
| `H` | 小时 | 每小时 |
| `T` 或 `min` | 分钟 | 每分钟 |

用 `pd.tseries.offsets.BDay()` 可自定义节假日。


## 案例：销售额按小时重采样

`resample()` 把高频数据聚合到低频（每小时 → 每天）。`between_time()` 筛选特定时段。

```python
s.resample('D').sum()                # 按天汇总
s.between_time('08:00', '22:00')     # 只取营业时间
```


In [ ]:
# 1. 生成模拟数据
# 创建一个从 2023-10-01 开始，频率为 1小时 的时间索引，共 72 小时（3天）
dates = pd.date_range(start='2023-10-01', periods=72, freq='h')

# 使用列表推导式生成符合业务逻辑的销售额
# 如果小时数在 8点到22点之间(不含22点)，生成大额销售；否则生成小额销售
sales_data = [
    np.random.randint(100, 500) if 8 <= t.hour < 22 else np.random.randint(0, 50)
    for t in dates
]

# 创建 Series
s = pd.Series(sales_data, index=dates)
s.name = "Hourly_Sales"

print("--- 原始数据预览 ---")
print(s.head(10))
print("...\n")


# 2. 按天重采样计算每日总销售额
# 'D' 代表按天聚合，sum() 代表求和
daily_total = s.resample('D').sum()

print("--- 每日总销售额 ---")
print(daily_total)
print("\n")

# 3. 计算营业时间 vs 非营业时间比例
# 利用 index 的 hour 属性进行布尔筛选
# 注意：8:00-22:00 通常指包含8点整，不包含22点整（即21:00是最后一个营业小时）
is_business_hours = (s.index.hour >= 8) & (s.index.hour < 22)

# 分别求和
business_sales = s[is_business_hours].resample('D').sum()
non_business_sales = s[~is_business_hours].resample('D').sum() #~表示逻辑非

# 计算比例：营业 / 非营业
# 为了防止除以0报错，可以用 fillna(0) 处理没有非营业数据的极端情况
ratio = business_sales / non_business_sales.fillna(1) # 这里填1是为了演示，实际可能填0或忽略
#fillna()，当你的数据中存在 NaN时，你可以用它指定一个值来替换这些空缺

result_ratio = pd.DataFrame({
    '营业总额': business_sales,
    '非营业总额': non_business_sales,
    '销额比例': ratio
})

print("--- 每天营业与非营业销售额比例 ---")
print(result_ratio)
print("\n")

# 4. 找出销售额最高的 3 个小时
# nlargest(n) 是获取前N大最高效的方法 series与dataframe都有
# nsmallest同理
top_3_hours = s.nlargest(3)

print("--- 销售额最高的 3 个小时 ---")
print(top_3_hours)